# 11. 침묵 vs 유저 반응 그룹 비교 분석

**분석 목적:** 리뷰 10개 미만인 `침묵` 그룹과 리뷰 10~49개인 `유저 반응` 그룹을 비교해, 유저의 첫 반응을 이끌어내기 위한 최소한의 기획적 조건을 탐색한다.

**핵심 질문:** 리뷰 10개 미만과 10~49개 게임은 장르·가격에서 어떻게 다른가?

**사용 데이터**
- `data/preprocessed/steam_indie_games.csv`: 유저 반응 그룹, 리뷰 10~49개
- `data/preprocessed/steam_indie_games_silence.csv`: 침묵 그룹, 리뷰 0~9개

**주의:** 현재 침묵 그룹은 Steam 태그 데이터가 미수집 상태이므로, 태그 Lift 분석은 데이터 가용성 진단만 수행한다.

In [1]:
import ast
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

## 1. 데이터 로드 및 그룹 변수 생성

In [2]:
DATA_DIR = Path("../../../data/preprocessed")
RESPONSE_PATH = DATA_DIR / "steam_indie_games.csv"
SILENCE_PATH = DATA_DIR / "steam_indie_games_silence.csv"

if not RESPONSE_PATH.exists():
    raise FileNotFoundError(f"유저 반응 데이터가 없습니다: {RESPONSE_PATH}")
if not SILENCE_PATH.exists():
    raise FileNotFoundError(
        f"침묵 그룹 데이터가 없습니다: {SILENCE_PATH}. "
        "00_preprocessing.ipynb를 먼저 실행해 생성하세요."
    )

df_response = pd.read_csv(RESPONSE_PATH)
df_response = df_response[df_response["total_reviews"].between(10, 49)].copy()
df_silence = pd.read_csv(SILENCE_PATH)

df_response["response_group"] = "유저 반응 (리뷰 10~49개)"
df_silence["response_group"] = "침묵 (리뷰 <10개)"

df = pd.concat([df_response, df_silence], ignore_index=True)
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

TARGET_GENRES = ["Action", "Adventure", "Casual", "RPG", "Simulation", "Strategy", "Sports", "Racing"]
GROUP_ORDER = ["침묵 (리뷰 <10개)", "유저 반응 (리뷰 10~49개)"]
GROUP_COLOR = {
    "침묵 (리뷰 <10개)": "#C44E52",
    "유저 반응 (리뷰 10~49개)": "#4C72B0",
}

print(f"유저 반응 그룹: {len(df_response):,}개")
print(f"침묵 그룹     : {len(df_silence):,}개")
print(f"전체          : {len(df):,}개")

df.head()

유저 반응 그룹: 4,840개
침묵 그룹     : 6,676개
전체          : 11,516개


,appid,positive,negative,price,genres,total_reviews,name,developers,release_date,short_description,publishers,categories,windows,mac,linux,recommendations_total,achievements_total,owners_lower,owners_higher,tags,response_group
0,318840,43,1,19.99,"['Casual', 'Indie']",44,Tempopo,Witch Beam,2025-04-17,A psychedelic soundscape in the sky. Puzzle th...,CULT Games,"Single-player, Steam Achievements, Full contro...",True,True,True,NaN,25.0,0,20000,"{""3D"": 310, ""Cozy"": 290, ""Cute"": 270, ""Indie"":...",유저 반응 (리뷰 10~49개)
1,331430,16,0,9.99,"['Adventure', 'Casual', 'Indie', 'RPG', 'Strat...",16,Toby's Island,Mvisioning,2023-09-15,"A 2D RPG adventure involving monster-raising, ...",Mvisioning,"Single-player, Steam Achievements, Partial Con...",True,False,False,NaN,18.0,0,20000,"{""2D"": 135, ""RPG"": 143, ""Anime"": 93, ""Magic"": ...",유저 반응 (리뷰 10~49개)
2,348740,20,4,0.99,"['Action', 'Adventure', 'Indie', 'Strategy']",24,Abyss Cave,Piao Jingfu,2024-12-08,"As a member of an elite squad, you are tasked ...",Piao Jingfu,"Single-player, Steam Achievements, Steam Tradi...",True,False,False,NaN,16.0,20000,50000,"{""Indie"": 21, ""Action"": 21, ""Strategy"": 22, ""A...",유저 반응 (리뷰 10~49개)
3,348930,11,0,19.99,"['Action', 'Adventure', 'Indie', 'RPG']",11,Lone Wolf,Play-Em,2024-12-13,Lone Wolf: A Post Apocalyptic Beat-Em Up Role ...,Play-Em,"Single-player, Multi-player, PvP, Shared/Split...",True,False,False,NaN,33.0,20000,50000,"{""PvP"": 55, ""RPG"": 76, ""2.5D"": 144, ""Co-op"": 1...",유저 반응 (리뷰 10~49개)
4,410000,20,5,4.99,"['Indie', 'Racing', 'Sports', 'Strategy']",25,Chalo Chalo,"Tomasz Kaye, Richard Boeser",2024-01-18,Chalo Chalo offers slow tactical racing for 3 ...,"Tomasz Kaye, Richard Boeser","Multi-player, PvP, Shared/Split Screen PvP, Sh...",True,False,False,NaN,6.0,0,20000,"{""Indie"": 22, ""Racing"": 23, ""Sports"": 20, ""Str...",유저 반응 (리뷰 10~49개)


## 2. 그룹 분포 요약

먼저 2023~2025년 필터를 통과한 전체 게임 중 침묵 그룹과 유저 반응 그룹이 어느 정도 비중인지 확인한다.

In [3]:
summary = (
    df.groupby("response_group")
    .agg(
        game_count=("appid", "nunique"),
        median_reviews=("total_reviews", "median"),
        avg_reviews=("total_reviews", "mean"),
        median_price=("price", "median"),
        tag_available=("tags", lambda s: s.notna().sum() if "tags" in df.columns else 0),
    )
    .reindex(GROUP_ORDER)
    .reset_index()
)
summary["ratio"] = summary["game_count"] / summary["game_count"].sum() * 100
summary.round(2)

,response_group,game_count,median_reviews,avg_reviews,median_price,tag_available,ratio
0,침묵 (리뷰 <10개),6676,3.0,3.78,3.99,6676,57.97
1,유저 반응 (리뷰 10~49개),4840,19.0,22.51,4.99,3001,42.03


In [4]:
fig = px.bar(
    summary,
    x="response_group",
    y="game_count",
    color="response_group",
    color_discrete_map=GROUP_COLOR,
    text="ratio",
    title="침묵 vs 유저 반응 그룹 규모",
    labels={"response_group": "그룹", "game_count": "게임 수"},
    category_orders={"response_group": GROUP_ORDER},
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(template="plotly_white", showlegend=False, width=850, height=480)
fig.show()

**해석:** 이 표와 그래프는 분석 필터를 통과한 최근 인디게임 중 “리뷰 10~49개”라는 첫 반응 기준을 넘지 못한 게임의 규모를 보여준다. 침묵 그룹 비중이 높을수록 출시 이후 첫 유저 반응을 만드는 것 자체가 중요한 과제임을 의미한다.

## 분석 1. 주요 장르 분포 비교

장르는 다중 장르를 모두 반영하기 위해 explode 방식으로 집계한다. 비교 기준은 절대 빈도가 아니라 각 그룹 내 장르 등장 비율이다.

In [5]:
def parse_genres(value: str) -> list[str]:
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(item).strip() for item in parsed if str(item).strip()]
    except (ValueError, SyntaxError):
        pass
    return [item.strip() for item in str(value).split(",") if item.strip()]


df["genre_list"] = df["genres"].apply(parse_genres)

genre_df = (
    df[df["genre_list"].map(len) > 0]
    .explode("genre_list")
    .rename(columns={"genre_list": "genre"})
)
genre_df["genre"] = genre_df["genre"].astype(str).str.strip()
genre_df = genre_df[genre_df["genre"].isin(TARGET_GENRES)].copy()

genre_counts = (
    genre_df.groupby(["response_group", "genre"])
    .agg(game_count=("appid", "nunique"))
    .reset_index()
)
group_totals = df.groupby("response_group")["appid"].nunique().rename("group_total")
genre_counts = genre_counts.merge(group_totals.reset_index(), on="response_group", how="left")
genre_counts["genre_rate"] = genre_counts["game_count"] / genre_counts["group_total"] * 100

genre_counts.head()

,response_group,genre,game_count,group_total,genre_rate
0,유저 반응 (리뷰 10~49개),Action,2197,4840,45.392562
1,유저 반응 (리뷰 10~49개),Adventure,2441,4840,50.433884
2,유저 반응 (리뷰 10~49개),Casual,2304,4840,47.603306
3,유저 반응 (리뷰 10~49개),RPG,874,4840,18.057851
4,유저 반응 (리뷰 10~49개),Racing,178,4840,3.677686


In [6]:
fig = px.bar(
    genre_counts,
    x="genre",
    y="genre_rate",
    color="response_group",
    barmode="group",
    color_discrete_map=GROUP_COLOR,
    title="침묵 vs 유저 반응 그룹의 주요 장르 비율",
    labels={"genre": "장르", "genre_rate": "그룹 내 장르 비율(%)", "response_group": "그룹"},
    category_orders={"response_group": GROUP_ORDER, "genre": TARGET_GENRES},
    hover_data={"game_count": ":,", "group_total": ":,"},
)
fig.update_layout(template="plotly_white", width=1000, height=520, yaxis=dict(ticksuffix="%"))
fig.show()

In [7]:
genre_pivot = (
    genre_counts.pivot(index="genre", columns="response_group", values="genre_rate")
    .reindex(TARGET_GENRES)
    .fillna(0)
)
genre_pivot["유저반응-침묵 차이(%p)"] = genre_pivot["유저 반응 (리뷰 10~49개)"] - genre_pivot["침묵 (리뷰 <10개)"]
genre_pivot.sort_values("유저반응-침묵 차이(%p)", ascending=False).round(2)

response_group,유저 반응 (리뷰 10~49개),침묵 (리뷰 <10개),유저반응-침묵 차이(%p)
genre,,,
Adventure,50.43,42.91,7.52
Simulation,22.73,18.51,4.21
RPG,18.06,16.99,1.07
Sports,3.90,3.73,0.18
Action,45.39,45.67,-0.28
Racing,3.68,4.10,-0.43
Strategy,19.38,21.14,-1.76
Casual,47.60,53.76,-6.16


**해석:** `유저반응-침묵 차이(%p)`가 양수인 장르는 유저 반응 그룹에서 상대적으로 더 많이 나타난 장르다. 반대로 음수인 장르는 침묵 그룹에서 상대적으로 더 많이 나타난 장르이므로, 출시 전 차별화나 첫 노출 전략을 더 엄격히 검토할 필요가 있다.

## 분석 2. 태그 데이터 가용성 진단

원래 계획은 침묵 그룹과 유저 반응 그룹의 태그 Lift를 비교하는 것이었다. 하지만 현재 `steam_indie_tags.csv`는 유저 반응 그룹 중심으로 수집되어 있으며, 침묵 그룹의 태그가 아직 매칭되지 않는다.

따라서 이 섹션에서는 태그 분석을 바로 수행하지 않고, 그룹별 태그 가용성을 진단한다. 침묵 그룹 태그를 추가 수집한 뒤 동일 구조로 Lift 분석을 확장할 수 있다.

In [8]:
tag_availability = (
    df.groupby("response_group")
    .agg(
        game_count=("appid", "nunique"),
        tag_available=("tags", lambda s: s.notna().sum()),
    )
    .reindex(GROUP_ORDER)
    .reset_index()
)
tag_availability["tag_available_rate"] = tag_availability["tag_available"] / tag_availability["game_count"] * 100

tag_availability.round(2)

,response_group,game_count,tag_available,tag_available_rate
0,침묵 (리뷰 <10개),6676,6676,100.0
1,유저 반응 (리뷰 10~49개),4840,3001,62.0


In [9]:
fig = px.bar(
    tag_availability,
    x="response_group",
    y="tag_available_rate",
    color="response_group",
    color_discrete_map=GROUP_COLOR,
    text="tag_available_rate",
    title="그룹별 태그 데이터 가용률",
    labels={"response_group": "그룹", "tag_available_rate": "태그 보유율(%)"},
    category_orders={"response_group": GROUP_ORDER},
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(template="plotly_white", showlegend=False, width=850, height=480, yaxis=dict(ticksuffix="%", range=[0, 110]))
fig.show()

**해석:** 침묵 그룹의 태그 보유율이 0%라면 태그 Lift 비교를 수행하면 안 된다. 이 경우 태그 차이는 실제 시장 속성 차이가 아니라 수집 범위 차이를 반영한다. 침묵 그룹 appid에 대해 Steam 태그를 추가 수집한 뒤 `parse_tags()`와 `top_tags(n=5)` 방식으로 Lift 분석을 재실행하는 것이 적절하다.

## 분석 3. 출시 가격 비교

가격은 첫 반응을 만들기 위한 진입 장벽으로 작동할 수 있다. 무료/F2P는 전처리에서 제외했으므로, 여기서는 유료 게임의 가격 포지셔닝을 비교한다.

In [10]:
price_df = df.copy()
price_df["price"] = pd.to_numeric(price_df["price"], errors="coerce")
price_df = price_df.dropna(subset=["price"])
price_df = price_df[(price_df["price"] > 0) & (price_df["price"] <= 60)].copy()

PRICE_BINS = [0, 5, 10, 15, 20, 30, float("inf")]
PRICE_LABELS = ["~$5", "$5~10", "$10~15", "$15~20", "$20~30", "$30+"]
price_df["price_range"] = pd.cut(price_df["price"], bins=PRICE_BINS, labels=PRICE_LABELS, right=True)

price_summary = (
    price_df.groupby("response_group")
    .agg(
        game_count=("appid", "nunique"),
        median_price=("price", "median"),
        avg_price=("price", "mean"),
        q1_price=("price", lambda s: s.quantile(0.25)),
        q3_price=("price", lambda s: s.quantile(0.75)),
    )
    .reindex(GROUP_ORDER)
    .reset_index()
)

price_summary.round(2)

,response_group,game_count,median_price,avg_price,q1_price,q3_price
0,침묵 (리뷰 <10개),6661,3.99,5.19,1.99,5.99
1,유저 반응 (리뷰 10~49개),4821,4.99,6.61,2.99,8.99


In [11]:
fig = px.box(
    price_df,
    x="response_group",
    y="price",
    color="response_group",
    color_discrete_map=GROUP_COLOR,
    category_orders={"response_group": GROUP_ORDER},
    points=False,
    title="침묵 vs 유저 반응 그룹의 출시 가격 분포",
    labels={"response_group": "그룹", "price": "가격 (USD)"},
)
fig.update_layout(template="plotly_white", showlegend=False, width=850, height=520)
fig.update_yaxes(tickprefix="$", range=[0, 60])
fig.show()

In [12]:
price_range_counts = (
    price_df.groupby(["response_group", "price_range"], observed=True)
    .agg(game_count=("appid", "nunique"))
    .reset_index()
)
price_group_total = price_range_counts.groupby("response_group")["game_count"].transform("sum")
price_range_counts["ratio"] = price_range_counts["game_count"] / price_group_total * 100

fig = px.bar(
    price_range_counts,
    x="response_group",
    y="ratio",
    color="price_range",
    text="ratio",
    title="침묵 vs 유저 반응 그룹의 가격대 구성 비율",
    labels={"response_group": "그룹", "ratio": "비율(%)", "price_range": "가격대"},
    category_orders={"response_group": GROUP_ORDER, "price_range": PRICE_LABELS},
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="inside")
fig.update_layout(template="plotly_white", width=900, height=520, yaxis=dict(ticksuffix="%"))
fig.show()

**해석:** 침묵 그룹이 특정 저가 또는 고가 구간에 더 몰려 있다면, 가격 자체가 문제라기보다 해당 가격대에서 기대되는 콘텐츠 규모·완성도·장르 매력도를 충족하지 못했을 가능성을 점검해야 한다. 유저 반응 그룹에서 상대적으로 강한 가격대는 첫 반응 확보에 유리했던 가격 포지셔닝 후보로 볼 수 있다.

## 종합 해석 — 유저 첫 반응을 이끌어내기 위한 최소 기획 조건

| 분석 축 | 확인할 질문 | 실무적 해석 |
|---|---|---|
| 그룹 규모 | 침묵 그룹은 얼마나 큰가? | 리뷰 10~49개 반응 확보 자체가 첫 번째 장벽인지 판단 |
| 장르 | 침묵 그룹에 과대표현되는 장르는 무엇인가? | 해당 장르에서 차별화·초기 노출 전략 필요 |
| 태그 | 태그 데이터가 비교 가능한가? | 현재는 침묵 태그 미수집으로 추가 수집 필요 |
| 가격 | 침묵 그룹이 특정 가격대에 몰리는가? | 가격 대비 기대 콘텐츠 규모와 포지셔닝 점검 |

**결론 방향:** 첫 반응을 만들기 위한 최소 조건은 단순히 “좋은 게임”이 아니라, 장르 포지셔닝·가격 기대치·초기 노출 가능성이 함께 맞아야 한다. 현재 데이터에서는 장르와 가격 차이를 먼저 확인하고, 침묵 그룹 태그 수집 후 태그 포지셔닝까지 확장하는 순서가 안전하다.